# FrameCraft · Figma D2C Agent

作者：[Loveiuer](https://github.com/Loveiuer)

本毕业设计将单个 Figma Frame 转成 React + TypeScript 页面骨架，保留设计节点 ID，并提供组件计划、结构检查与导出产物。在线工作流由 DesignAnalyst、ComponentPlanner、CodeEngineer、Reviewer 四个 HelloAgents `SimpleAgent` 协作，确定性编译器负责生成代码。

本 Notebook 默认运行**离线 demo**，不调用模型、不读取 Figma、不自动安装依赖，也不加载任何环境文件。它验证设计导入、编译、检查和导出链路，不能证明四个真实 Agent 已执行。节点覆盖率不能代表视觉还原率，输出按钮默认禁用，业务交互需要另行实现。


## 运行准备

使用 Python 3.10+，先在项目目录按 README 安装 `requirements.txt`，再选择对应 Python 环境作为 Notebook 内核。Notebook 前端可使用 Jupyter 或 VS Code；命令行也可以顺序执行下列 Python 单元。

可从项目目录、Hello-Agents 仓库根目录或本地学习工作区运行。输出写入本项目的 `outputs/notebook-<时间戳>/`，已存在目录不会被覆盖。完整 React 构建另需 Node.js 20.19+ 和 npm；本 Notebook 不安装 npm 依赖，也不执行生成的前端代码。


In [ ]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

# 仅探测项目文件；不会读取 .env 或任何祖先目录的环境配置。
working_directory = Path.cwd().resolve()
project_root = None
for base in (working_directory, *working_directory.parents):
    candidates = (
        base,
        base / "Co-creation-projects" / "Loveiuer-FrameCraft",
        base / "code" / "chapter16" / "figma-d2c-agent",
    )
    for candidate in candidates:
        if all((candidate / marker).is_file() for marker in (
            "d2c/pipeline.py", "d2c/artifacts.py", "data/sample-design.json"
        )):
            project_root = candidate.resolve()
            break
    if project_root is not None:
        break
if project_root is None:
    raise FileNotFoundError("未找到 FrameCraft 项目，请从项目目录或仓库根目录运行。")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from d2c.pipeline import run_pipeline
from d2c.artifacts import write_result

print("项目目录已找到；运行模式固定为离线 demo。")


## 导入设计并执行离线流程

内置样例是包含文本、Auto Layout 与基础样式的 104 节点学习工作台。每个阶段会输出真实状态；demo 的模型调用数应为 0。


In [ ]:
sample = json.loads((project_root / "data" / "sample-design.json").read_text(encoding="utf-8"))

def print_event(event):
    print(f"[{event['stage']}/{event['status']}] {event['message']}")

result = run_pipeline(sample, mode="demo", on_event=print_event)
assert result["mode"] == "demo"
assert result["metrics"]["llm_calls"] == 0
assert result["review"]["passed"], "结构检查未通过，请检查下方报告。"
print("\n节点统计")
print(json.dumps(result["design"]["stats"], ensure_ascii=False, indent=2))


## 检查分析、组件计划与报告

下面的数据来自本次执行。组件计划将节点 ID 映射到语义标签；报告说明结构覆盖、警告和检查边界。分析与计划在 demo 中来自确定性规则，不是模型生成结果。


In [ ]:
for title, evidence in (
    ("设计分析", result["analysis"]),
    ("组件计划", result["plan"]),
    ("结构检查报告", result["review"]),
    ("运行指标", result["metrics"]),
):
    print("\n" + title)
    print(json.dumps(evidence, ensure_ascii=False, indent=2))


## 导出 React 项目与运行证据

导出目录包含 `preview.html`、React / TypeScript / CSS 源码、设计模型、组件计划、执行轨迹和报告。可以用浏览器手动打开静态预览；静态预览成功不代表 React 工程已完成构建。需要构建时，进入导出目录按其中的 README 执行 npm 命令。


In [ ]:
timestamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
destination = project_root / "outputs" / f"notebook-{timestamp}"
exported_path = write_result(result, destination)
assert (exported_path / "preview.html").is_file()
assert (exported_path / "src" / "App.tsx").is_file()
print("导出完成：" + str(exported_path.relative_to(project_root)))
print("静态预览：" + str((exported_path / "preview.html").relative_to(project_root)))
print("导出文件")
for file_path in sorted(exported_path.rglob("*")):
    if file_path.is_file():
        print("  " + str(file_path.relative_to(exported_path)))


## 可选：真实四 Agent 流程

本段是手动操作示例，不是可执行单元，“运行全部”不会发起模型请求。

先将本项目的 `.env.example` 复制为本项目的 `.env`，填写模型配置。在线模式会把设计摘要与前序分析发送给配置的模型服务，产生费用并受网络、限流和 JSON 格式兼容性影响。建议先使用 6 节点的小样例；不能从一次小样例成功推断大型设计稳定。

如需真实验证，可手动新建代码单元，复制下列内容。仅显式读取本项目 `.env`，不会搜索祖先目录；不要把密钥或带密钥的日志保存在 Notebook 中。

```python
from dotenv import load_dotenv

load_dotenv(project_root / ".env", override=False)
small_sample = json.loads(
    (project_root / "data" / "smoke-design.json").read_text(encoding="utf-8")
)
live_result = run_pipeline(small_sample, mode="live", on_event=print_event)
print(json.dumps(live_result["metrics"], ensure_ascii=False, indent=2))
live_destination = project_root / "outputs" / (
    "notebook-live-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
)
write_result(live_result, live_destination)
```

真实 Figma 导入还需要有权访问的 Frame 链接和 Token，使用方式见 [README](README.md)。已有验证记录见 [docs/validation.json](docs/validation.json)，架构与限制见 [docs/architecture.md](docs/architecture.md)。真实 Figma 联调和大样例在线稳定性仍是未完成的验收范围。
